In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # wait for few minutes
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory #in built memory in langchain --
from langchain_core.messages import HumanMessage, AIMessage # by default your LLM try --> input --> human or AI meeage
from langchain_core.runnables.history import RunnableWithMessageHistory # wait for 15 mint
load_dotenv()

True

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini")
llm.invoke('hi')


AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a17b1ca31d', 'id': 'chatcmpl-E1MJu1BsVkyLIdoK2xl8jquHjcMb8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f5e42-586c-7370-b7b9-e1baaa6e3219-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
history = InMemoryChatMessageHistory()
history.messages



[]

In [ ]:
# history.add_user_message("I have a billing issue")

In [ ]:
# history.messages

[HumanMessage(content='I have a billing issue', additional_kwargs={}, response_metadata={})]

In [ ]:
# history.add_ai_message("Undrstood you query but I can thelp you")


In [4]:
# history.messages

In [6]:
# MessagesPlaceholder
# a placeholder inside your prompt ( chatprompt templates)

prompt = ChatPromptTemplate.from_messages([
    ("system","You are a professional email support agent"),
    MessagesPlaceholder(variable_name="history"), # slot for past message
    ("human","{input}") # current user message

    ])

prompt

ChatPromptTemplate(input_variables=['history', 'input'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_c

In [7]:
past_messages = [
    HumanMessage(content="Billing complaint from John — invoice #1042, overcharged $250."),
    AIMessage(content="Classified: Billing — High Priority. SLA: 2 hours."),
]


prompt.format_messages(
    history = past_messages,
    input="Draft an aplology email for John"
)

[SystemMessage(content='You are a professional email support agent', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Billing complaint from John — invoice #1042, overcharged $250.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Classified: Billing — High Priority. SLA: 2 hours.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Draft an aplology email for John', additional_kwargs={}, response_metadata={})]

In [8]:
# step 1 : defined your propt
prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify complaints as: Billing / Technical / General.
Priority: High (financial impact > $100 or urgent) / Medium / Low.
Remember all customer details throughout the conversation."""),
    MessagesPlaceholder(variable_name="history"),  # ← past turns injected here
    ("human", "{input}"),                           # ← new message here
])


# step 2 LCEL
chain = prompt | llm | StrOutputParser()

In [9]:
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] =InMemoryChatMessageHistory()
    return store[session_id]

In [12]:
get_session_history("3213")

InMemoryChatMessageHistory(messages=[])

In [14]:
email_assistant= RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"

)

/Users/rahultiwari/Documents/02_Freelancing/Hachion_batch/ai_engineering_19th_may/ai-env/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [15]:
# rahul will login to the system
cfg = {"configurable":{"session_id":"123rahul"}}


r1 = email_assistant.invoke({"input": "I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250."},
                            config=cfg
)

In [17]:
print(r1) # r1

Complaint Classification: Billing  
Priority: High  

Customer Details:  
Name: John  
Invoice Number: #1042  
Incorrect Charge: $500 instead of $250  


In [18]:
store

{'1234': InMemoryChatMessageHistory(messages=[]),
 '3213': InMemoryChatMessageHistory(messages=[]),
 '123rahul': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Complaint Classification: Billing  \nPriority: High  \n\nCustomer Details:  \nName: John  \nInvoice Number: #1042  \nIncorrect Charge: $500 instead of $250  ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}

In [19]:
r2 = email_assistant.invoke(
    {"input": "What category does his complaint fall under?"},
    config=cfg
)

print(r2)

John's complaint falls under the category of Billing.


In [22]:
cfg = {"configurable":{"session_id":"123rahul"}}

r2 = email_assistant.invoke(
    {"input": "What category does his complaint fall under?"},
    config=cfg
)
r2

"John's complaint falls under the category of Billing."

In [21]:
store

{'1234': InMemoryChatMessageHistory(messages=[]),
 '3213': InMemoryChatMessageHistory(messages=[]),
 '123rahul': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Complaint Classification: Billing  \nPriority: High  \n\nCustomer Details:  \nName: John  \nInvoice Number: #1042  \nIncorrect Charge: $500 instead of $250  ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content="John's complaint falls under the category of Billing.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]),
 'shubham_321': InMemoryChatMessageHistory(messages=[HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, re